In [14]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

**DATA GENERATION**
Generate synthetic projectile-motion dataset

In [15]:
g         = 9.81
v0        = 50.0
theta     = np.radians(45.0)
T_flight  = 2 * v0 * np.sin(theta) / g
N         = 500
t_all     = np.linspace(0, T_flight, N)

x_true = v0 * np.cos(theta) * t_all
y_true = v0 * np.sin(theta) * t_all - 0.5 * g * t_all**2

# Add sensor noise
x_noisy = x_true + np.random.normal(0, 1.0, N)
y_noisy = y_true + np.random.normal(0, 1.0, N)

***NORMALISE***

In [16]:
t_min, t_max = t_all.min(), t_all.max()
x_min, x_max = x_true.min(), x_true.max()
y_min, y_max = y_true.min(), y_true.max()

t_norm = (t_all    - t_min) / (t_max - t_min + 1e-8)
x_norm = (x_noisy  - x_min) / (x_max - x_min + 1e-8)
y_norm = (y_noisy  - y_min) / (y_max - y_min + 1e-8)

T_tensor  = torch.tensor(t_norm, dtype=torch.float32).unsqueeze(1)
XY_tensor = torch.tensor(np.stack([x_norm, y_norm], axis=1), dtype=torch.float32)

split    = int(0.8 * N)
X_train  = T_tensor[:split];   y_train = XY_tensor[:split]
X_test   = T_tensor[split:];   y_test  = XY_tensor[split:]

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

**MODEL**

In [17]:
class TrajectoryMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1,   64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64,   2)
        )
    def forward(self, x):
        return self.net(x)

model     = TrajectoryMLP()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)

**TRAINING**

In [18]:
EPOCHS       = 300
train_losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for t_b, xy_b in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(t_b), xy_b)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(t_b)
    epoch_loss /= len(X_train)
    train_losses.append(epoch_loss)
    scheduler.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch}/{EPOCHS}  MSE: {epoch_loss:.6f}")

Epoch 50/300  MSE: 0.001445
Epoch 100/300  MSE: 0.001085
Epoch 150/300  MSE: 0.000931
Epoch 200/300  MSE: 0.000702
Epoch 250/300  MSE: 0.000677
Epoch 300/300  MSE: 0.000622


**EVALUATE**

In [19]:
model.eval()
with torch.no_grad():
    pred_norm = model(X_test).numpy()

x_pred = pred_norm[:, 0] * (x_max - x_min) + x_min
y_pred = pred_norm[:, 1] * (y_max - y_min) + y_min
x_act  = x_true[split:]
y_act  = y_true[split:]

mae_x  = np.mean(np.abs(x_pred - x_act))
mae_y  = np.mean(np.abs(y_pred - y_act))
rmse_x = np.sqrt(np.mean((x_pred - x_act)**2))
rmse_y = np.sqrt(np.mean((y_pred - y_act)**2))
print(f"\nMAE  → x: {mae_x:.2f} m   y: {mae_y:.2f} m")
print(f"RMSE → x: {rmse_x:.2f} m   y: {rmse_y:.2f} m")


MAE  → x: 7.00 m   y: 16.73 m
RMSE → x: 7.42 m   y: 19.12 m


Plot figures

In [20]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Q1 – Ballistic Trajectory Prediction (MLP)", fontsize=13, fontweight='bold')

axes[0].plot(x_true, y_true, 'b-',  lw=2.5, label='True Trajectory')
axes[0].plot(x_noisy[:split], y_noisy[:split], 'g.', alpha=0.2, ms=3, label='Noisy Training Data')
axes[0].plot(x_pred, y_pred,  'r--', lw=2,   label='NN Prediction')
axes[0].set_xlabel('x (m)'); axes[0].set_ylabel('y (m)')
axes[0].set_title('True vs Predicted Trajectory')
axes[0].set_ylim(bottom=0); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].semilogy(train_losses, 'k-', lw=1.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE Loss (log)')
axes[1].set_title('Training Loss'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('q1_trajectory_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Done. Figure saved as q1_trajectory_results.png")

Done. Figure saved as q1_trajectory_results.png
